# WavLM Evaluation - Embedding-Based Classification (Machine Learning)

This notebook evaluates the predictive power of WavLM embeddings for speaker traits using various machine learning models (Logistic Regression, MLP, HGBT).

**Validation Strategy**: StratifiedGroupKFold (n=10) by `speaker_id` to ensure health status balance and prevent data leakage.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Pandas display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Add root directory to sys.path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from embeddings_eval.data_loader import load_embeddings, load_metadata
from embeddings_eval.analyzer import WavLMAnalyzer
from embeddings_eval.reporter import WavLMReporter
from embeddings_eval.constants import GROUP_DDK

In [ ]:
DATA_DIR = "../datalocal/PC-GITA_v260210_24kHz/speaker_embeddings/wavLM"
META_PATH = "../datalocal/PC-GITA_v260210_24kHz/_metadata/PCGITAtoPD_mapping.csv"

print("Loading data and metadata...")
metadata = load_metadata(META_PATH)
embeddings = load_embeddings(DATA_DIR, metadata=metadata)
analyzer = WavLMAnalyzer(embeddings)

print(f"Loaded {len(embeddings)} samples.")

In [ ]:
# Data preparation for scikit-learn
X = np.stack([e.vector.cpu().numpy() for e in embeddings])
y_sex = np.array([e.sex for e in embeddings])
y_age = np.array([e.age for e in embeddings])
y_status = np.array([e.health_status for e in embeddings])
y_hy = np.array([e.hy for e in embeddings])
groups = np.array([e.speaker_id for e in embeddings])
task_groups = np.array([e.group for e in embeddings])

cv = StratifiedGroupKFold(n_splits=10)

def run_ml_experiment(X, y, groups, model_name='lr', task='clf', label='Experiment', return_proba=False):
    if task == 'clf':
        preds = np.zeros_like(y, dtype=object)
        probas = None
        if return_proba: probas = np.zeros((len(y), 2))
    else:
        preds = np.zeros_like(y, dtype=float)
    
    folds = list(cv.split(X, y, groups))
    for train_idx, test_idx in tqdm(folds, desc=f"Training {label} ({model_name.upper()})"):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        if task == 'clf':
            if model_name == 'lr': model = LogisticRegression(max_iter=1000)
            elif model_name == 'mlp': model = MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500)
            elif model_name == 'hgbt': model = HistGradientBoostingClassifier()
            
            model.fit(X_train, y_train)
            preds[test_idx] = model.predict(X_test)
            if return_proba: probas[test_idx] = model.predict_proba(X_test)
        else: # Regression
            if model_name == 'ridge': model = Ridge()
            elif model_name == 'hgbt': model = HistGradientBoostingRegressor()
            model.fit(X_train, y_train)
            preds[test_idx] = model.predict(X_test)
        
    if return_proba: return preds, probas, model.classes_
    return preds

## 1. Sex Classification (M vs F)
Comparing Logistic Regression, 3-layer MLP, and HistGradientBoosting.

In [ ]:
results_sex = {}
for m in ['lr', 'mlp', 'hgbt']:
    p = run_ml_experiment(X, y_sex, groups, model_name=m, task='clf', label='Sex')
    results_sex[m] = p
    acc = accuracy_score(y_sex, p)
    print(f"\n--- Result for SEX ({m.upper()}) ---")
    print(f"Overall Accuracy: {acc*100:.2f}%")
    res_tmp = pd.DataFrame({'group': task_groups, 'true': y_sex, 'pred': p})
    res_tmp['is_correct'] = res_tmp['true'] == res_tmp['pred']
    display(res_tmp.groupby('group')['is_correct'].mean() * 100)

# Using LR for breakdown
preds_sex = results_sex['lr']
res_sex = pd.DataFrame({'speaker_id': groups, 'status': y_status, 'group': task_groups, 'true': y_sex, 'pred': preds_sex})
res_sex['is_correct'] = res_sex['true'] == res_sex['pred']

print("\nDetailed Speaker Breakdown (Sex - LR):")
speaker_sex_res = []
for (sid, status, true_sex), s_data in res_sex.groupby(['speaker_id', 'status', 'true']):
    total = len(s_data)
    correct = s_data['is_correct'].sum()
    accuracy = (correct / total) * 100
    wrong_assignments = s_data[s_data['is_correct'] == False]
    most_common_confusion = "None" if wrong_assignments.empty else Counter(wrong_assignments['pred']).most_common(1)[0][0]
    speaker_sex_res.append({'speaker': sid, 'status': status, 'true sex': true_sex, 'accuracy %': accuracy, 'misclassified count': total - correct, 'most common confusion': most_common_confusion})
speaker_sex_df = pd.DataFrame(speaker_sex_res)
display(speaker_sex_df)

print("\nTOP 5 Misclassified Speakers (Sex):")
top5_sex = speaker_sex_df.sort_values(by='accuracy %').head(5)
for _, row in top5_sex.iterrows():
    print(f"{row['speaker']} ({row['accuracy %']:.1f}%, {row['misclassified count']} misclassified, most common confusion: {row['most common confusion']})")

## 2. Age Prediction (Years)
Metric: Mean Absolute Error (MAE).

In [ ]:
results_age = {}
for m in ['ridge', 'hgbt']:
    p = run_ml_experiment(X, y_age, groups, model_name=m, task='reg', label='Age')
    results_age[m] = p
    err = np.abs(y_age - p)
    print(f"\n--- Result for AGE ({m.upper()}) ---")
    print(f"Overall MAE: {err.mean():.2f} years (Var: {err.var():.2f})")
    res_tmp = pd.DataFrame({'group': task_groups, 'abs_error': err})
    display(res_tmp.groupby('group')['abs_error'].agg(['mean', 'var']))

p_hgbt = results_age['hgbt']
res_age = pd.DataFrame({'speaker_id': groups, 'status': y_status, 'true': y_age, 'pred': p_hgbt})
res_age['abs_error'] = np.abs(res_age['true'] - res_age['pred'])
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); sns.histplot(data=res_age.drop_duplicates('speaker_id'), x='true', hue='status', multiple="dodge", shrink=.8); plt.title("Age Distribution by Status"); plt.xlabel("True Age")
plt.subplot(1, 2, 2); sns.histplot(data=res_age, x='abs_error', hue='status', element="step"); plt.title("Age Prediction Error Distribution (HGBT)"); plt.xlabel("Absolute Error (Years)")
plt.tight_layout(); plt.show()

## 3. PD vs HC Detection

**NOTE**: DDK task is excluded to avoid bias (HC group lacks DDK recordings).

In [ ]:
mask_no_ddk = (task_groups != GROUP_DDK)
X_f, y_s_f, y_hy_f, gr_f, tg_f = X[mask_no_ddk], y_status[mask_no_ddk], y_hy[mask_no_ddk], groups[mask_no_ddk], task_groups[mask_no_ddk]

results_status, probas_status, classes_status = {}, {}, None
for m in ['lr', 'mlp', 'hgbt']:
    p, prob, cl = run_ml_experiment(X_f, y_s_f, gr_f, model_name=m, task='clf', label='PD/HC', return_proba=True)
    results_status[m], probas_status[m], classes_status = p, prob, cl
    acc = accuracy_score(y_s_f, p)
    print(f"\n--- Result for PD/HC ({m.upper()}) ---"); print(f"Overall Accuracy: {acc*100:.2f}%")
    res_tmp = pd.DataFrame({'group': tg_f, 'true': y_s_f, 'pred': p})
    res_tmp['is_correct'] = res_tmp['true'] == res_tmp['pred']
    display(res_tmp.groupby('group')['is_correct'].mean() * 100)

print("\n--- Misclassified Speakers (PD/HC - LR) ---")
res_lr = pd.DataFrame({'speaker_id': gr_f, 'status': y_s_f, 'true': y_s_f, 'pred': results_status['lr'], 'hy': y_hy_f})
res_lr['is_correct'] = res_lr['true'] == res_lr['pred']
mis_speakers = []
for (sid, status, hy), s_data in res_lr.groupby(['speaker_id', 'status', 'hy']):
    total = len(s_data); incorrect = (s_data['is_correct'] == False).sum()
    if incorrect > 0:
        mis_speakers.append({'speaker_id': sid, 'status': status, 'H/Y': hy, 'wrong samples': incorrect, 'total samples': total, 'accuracy %': ((total-incorrect)/total)*100})
mis_df = pd.DataFrame(mis_speakers)
if not mis_df.empty: display(mis_df.sort_values(by='accuracy %'))
else: print("No misclassified speakers found!")

### 3.1 Aggregated PD/HC Detection (Per-Speaker Summary)

In [ ]:
def get_styled_summary(res_df, prob_matrix, classes, model_label):
    df = res_df.copy()
    for i, c_n in enumerate(classes): df[f'prob_{c_n}'] = prob_matrix[:, i]
    agg_r = []
    for (sid, gid, status, hy), g_data in df.groupby(['speaker_id', 'group', 'status', 'hy']):
        avg_p = [g_data[f'prob_{c}'].mean() for c in classes]
        agg_r.append({'speaker_id': sid, 'group': gid, 'status': status, 'H/Y': hy, 'avg_p_pred': classes[np.argmax(avg_p)]})
    agg_df = pd.DataFrame(agg_r)
    s_pivot = agg_df.pivot(index=['speaker_id', 'status', 'H/Y'], columns='group', values='avg_p_pred')
    overall_r = []
    for (sid, status), s_data in df.groupby(['speaker_id', 'status']):
        avg_p = [s_data[f'prob_{c}'].mean() for c in classes]
        overall_r.append({'speaker_id': sid, 'status': status, 'Overall Classification': classes[np.argmax(avg_p)], 'Overall Score': np.max(avg_p)})
    ov_df = pd.DataFrame(overall_r).set_index(['speaker_id', 'status'])
    f_df = ov_df.join(s_pivot.reset_index(level='H/Y'))
    def color_f(row):
        target = row.name[1]; ov_p = row['Overall Classification']
        gr_cols = [c for c in row.index if c in ['monologue', 'readtext', 'sentence', 'words']]
        all_c = all(row[c] == target for c in gr_cols if pd.notna(row[c]))
        styles = [''] * len(row); ov_pos = row.index.get_loc('Overall Classification')
        if ov_p != target: styles[ov_pos] = 'color: red; font-weight: bold'
        elif all_c: styles[ov_pos] = 'color: green; font-weight: bold'
        for c in gr_cols: 
            if pd.notna(row[c]) and row[c] != target: styles[row.index.get_loc(c)] = 'background-color: orange'
        return styles
    disp_df = f_df.copy(); disp_df['Overall Classification'] = f_df.apply(lambda x: f"{x['Overall Classification']} ({x['Overall Score']:.2f})", axis=1)
    disp_df = disp_df.drop(columns=['Overall Score'])
    print(f"\n--- Per-Speaker Summary ({model_label}) ---"); display(disp_df.style.apply(color_f, axis=1))

for m in ['lr', 'hgbt']:
    res_t = pd.DataFrame({'speaker_id': gr_f, 'status': y_s_f, 'group': tg_f, 'true': y_s_f, 'pred': results_status[m], 'hy': y_hy_f})
    get_styled_summary(res_t, probas_status[m], classes_status, m.upper())